In [0]:
from delta.tables import DeltaTable
from pyspark.sql.dataframe import DataFrame

def upsert_data(df_source: DataFrame, full_table_target_name: str, join_key_condition: list):
    """
    Realiza o MERGE. 
    df_source: O próprio DataFrame do PySpark processado.
    full_table_target_name: O nome completo no formato 'catalog.schema.tabela'.
    join_key_condition: Uma lista de strings com os nomes das primary_key.
    """

    spark = df_source.sparkSession 
    
    table_exist = spark.catalog.tableExists(full_table_target_name)

    if not table_exist:
        print(f" Table {full_table_target_name} DOES NOT exist. Initializing full load")
        df_source.write.format("delta").mode("overwrite").saveAsTable(full_table_target_name)
        print(" Full load complete.")

    else:
        print(f"Table {full_table_target_name} exists. Initializing merge")
        

        deltaTableTarget = DeltaTable.forName(spark, full_table_target_name)

    
        join_condition = " AND ".join([f"source.{k} = target.{k}" for k in join_key_condition])

        deltaTableTarget.alias("target").merge(
            df_source.alias("source"),
            join_condition
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        
        last_operation = deltaTableTarget.history(1).collect()
        metrics = last_operation["operationMetrics"]
        
        rows_insert = metrics.get("numTargetRowsInserted", "0")
        rows_update = metrics.get("numTargetRowsUpdated", "0")
        
        print(f" MERGE updated successfully")
        print(f" Rows inserted: {rows_insert}")
        print(f" Rows updated: {rows_update}")